In [1]:
import pandas as pd
import config as cfg
import os
import glob
from sklearn.metrics import roc_auc_score
import torch

import config
from utils.metrics import compute_AUCs_for_multiple_endpoints
from utils.calibration_plots import adaptive_make_calibration_plots

C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_6212\2284837203.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was too old on your system - pyarrow 10.0.1 is the current minimum supported version as of this release.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:

csv_path = "datasets/MT_dataset/stratified_sampling_test_542.csv"
#csv_path = "//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/Daniel MacRae/NTCP_Multitox/stratified_sampling_test_02_09.csv"
df = pd.read_csv(csv_path, delimiter=";")
print(len(df))
df['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df["PatientID"]]




1090


In [2]:
#df_CITOR_REDCAP = pd.read_excel("//zkh/appdata/RTDicom/Projectline_HNC_modelling/Users/Daniel MacRae/CITOR_REDCAP_clinical_data_important_variables_combined.xlsx")

redcap_dir = "C:/Users/d.c.macrae/Documents/CITOR_REDCAP_clinical_data_important_variables_combined.xlsx"
df_CITOR_REDCAP = pd.read_excel(redcap_dir)

df_CITOR_REDCAP['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df_CITOR_REDCAP["UMCG"]]
df_chemo_data = df_CITOR_REDCAP[["PatientID", "Modality_adjusted", "LEEFTIJD"]].copy()


df_chemo_data["Chemotherapy"] = df_chemo_data["Modality_adjusted"].str.contains("chemoradiation", case=False).astype(int)
df_chemo_data["Biotherapy"] = df_chemo_data["Modality_adjusted"].str.contains("bioradiation", case=False).astype(int)


df_chemo_only = df_chemo_data[["PatientID", "Chemotherapy"]].copy()



#chemo_patients = df_chemo_data[df_chemo_data["Modality_adjusted"] == "Chemotherapy"].copy()
#chemo_patient_IDS = df_chemo_data["PatientID"].tolist()

In [3]:
df_strat_sampling = pd.read_csv("datasets/MT_dataset/stratified_sampling_test_101.csv", delimiter=";")
df_strat_sampling['PatientID'] = [str(item).zfill(cfg.patient_id_length) for item in df_strat_sampling["PatientID"]]

df_strat_sampling2 = pd.merge(df_strat_sampling, df_chemo_data, on="PatientID", how="left")
df_strat_sampling2


df_strat_sampling2["Age_binary"] = df_strat_sampling2["Age"] >= 70
df_strat_sampling2["Age_binary"] = df_strat_sampling2["Age_binary"].astype(int)
#df_strat_sampling2.to_csv("datasets/MT_dataset/stratified_sampling_test_542_chemo.csv", index=False, sep=";")

In [9]:
df_strat_sampling2[df_strat_sampling2["Split"] == 'test']["Chemotherapy"].value_counts(normalize=True)

Chemotherapy
0    0.637615
1    0.362385
Name: proportion, dtype: float64

In [8]:
df_strat_sampling2[df_strat_sampling2["Split"] != 'test']["Chemotherapy"].value_counts(normalize=True)

Chemotherapy
0    0.641055
1    0.358945
Name: proportion, dtype: float64

In [28]:
df_strat_sampling2["over_70"] = df_strat_sampling2["Age"].apply(lambda x: 1 if x >= 0.7 else 0)

comb_counts = df_strat_sampling2[["Chemotherapy", "over_70"]].groupby(["Chemotherapy", "over_70"]).value_counts()
print(comb_counts)
comb_counts = df_strat_sampling2[["Chemotherapy", "over_70"]].groupby(["Chemotherapy", "over_70"]).value_counts() / len(df_strat_sampling2)
print(comb_counts)


df_strat_sampling2_test = df_strat_sampling2[df_strat_sampling2["Split"] == "test"]
comb_counts_test = df_strat_sampling2_test[["Chemotherapy", "over_70"]].groupby(["Chemotherapy", "over_70"]).value_counts()
print(comb_counts_test)
comb_counts = df_strat_sampling2_test[["Chemotherapy", "over_70"]].groupby(["Chemotherapy", "over_70"]).value_counts()  / len(df_strat_sampling2_test)
print(comb_counts)

Chemotherapy  over_70
0             0          388
              1          310
1             0          385
              1            7
Name: count, dtype: int64
Chemotherapy  over_70
0             0          0.355963
              1          0.284404
1             0          0.353211
              1          0.006422
Name: count, dtype: float64
Chemotherapy  over_70
0             0          67
              1          60
1             0          89
              1           2
Name: count, dtype: int64
Chemotherapy  over_70
0             0          0.307339
              1          0.275229
1             0          0.408257
              1          0.009174
Name: count, dtype: float64


In [12]:
# load ensemble predictions

experiment_names = ["Best_DCNN_ECE", "Best_ResNet_ECE", "Best_ViT", "Best ResNeXt"]
model_names = ["dcnn_pooling", "resnet_lrelu", "ViT", "resnext_lrelu"]


experiment_names = ["DCNN_Chemo", "ResNet_Chemo"] #, "Best_ViT", "Best ResNeXt"]
model_names = ["dcnn_pooling", "resnet_lrelu"] #, "ViT", "resnext_lrelu"]


experiment_names = ["ResNet 10", "ResNet_50_1"] #, "Best_ViT", "Best ResNeXt"]
model_names = ["resnet_lrelu", "resnet_lrelu"] #, "ViT", "resnext_lrelu"]


experiment_names = ["TransRP_ResNet18_m2 features"] #, "Best_ViT", "Best ResNeXt"]
model_names = ["TransRP_ResNet18_m2"] #, "ViT", "resnext_lrelu"]


# experiment_names = [ "TransRP_DenseNet121_m2"] #, "Best_ViT", "Best ResNeXt"]
# model_names = ["TransRP_DenseNet121_m2"] #, "ViT", "resnext_lrelu"]
endpoints = ["Taste_M06"]
endpoints = config.endpoint_list


# LR MODEL
print("CITOR MODEL")
lr_ens_dir = glob.glob(f"experiments/{experiment_names[0]}/**/lr_ens_outputs.csv", recursive=True)[0]
df_lr_ens_preds = pd.read_csv(lr_ens_dir, delimiter=";")
df_lr_ens_preds["PatientID"] = [str(item).zfill(cfg.patient_id_length) for item in df_lr_ens_preds["PatientID"]]
df_lr_combined = pd.merge(df_lr_ens_preds, df_chemo_data, on="PatientID", how="left")
#df_lr_combined = pd.merge(df_lr_combined, df_chemo_data[["PatientID", "LEEFTIJD"]], on="PatientID", how="left")


chemo_ensemble_DL_predictions_list_dict = {}
chemo_true_labels_list_dict = {}
no_chemo_ensemble_DL_predictions_list_dict = {}
no_chemo_true_labels_list_dict = {}

for endpoint in endpoints:
        # drop all nones
        mask = df_lr_combined[endpoint+"_true"].isin([0, 1])
        #mask2 = auc_no_chemo[endpoint+"_true"].isin([0, 1])

        df_combined_temp = df_lr_combined[mask]

        df_no_chemo_preds = df_combined_temp[df_lr_combined["Chemotherapy"] == 0].copy()
        df_chemo_preds = df_combined_temp[df_lr_combined["Chemotherapy"] == 1].copy()

        chemo_ensemble_DL_predictions_list_dict[endpoint] = torch.tensor(df_chemo_preds[endpoint+"_pred"].tolist())
        chemo_true_labels_list_dict[endpoint] = torch.tensor(df_chemo_preds[endpoint+"_true"].tolist())
        no_chemo_ensemble_DL_predictions_list_dict[endpoint] = torch.tensor(df_no_chemo_preds[endpoint+"_pred"].tolist())
        no_chemo_true_labels_list_dict[endpoint] = torch.tensor(df_no_chemo_preds[endpoint+"_true"].tolist())
        
        auc_no_chemo = roc_auc_score(df_no_chemo_preds[endpoint+"_true"], df_no_chemo_preds[endpoint+"_pred"])
        auc_chemo = roc_auc_score(df_chemo_preds[endpoint+"_true"], df_chemo_preds[endpoint+"_pred"])
        print(f"Endpoint: {endpoint}, AUC no chemotherapy: {auc_no_chemo:.3f}, AUC yes chemotherapy: {auc_chemo:.3f}")

# make_calibration_plots(cfg, predictions_dict=no_chemo_ensemble_DL_predictions_list_dict, true_labels_dict=no_chemo_true_labels_list_dict, 
#                                         lr_predictions_dict=chemo_ensemble_DL_predictions_list_dict, lr_true_labels_dict=chemo_true_labels_list_dict, mode='calibration', set_name="Ensemble (Test Set)", 
#                                         #filename=os.path.join(cfg.exp_dir, 'calibration_plot_ensemble.png'))
#                                         filename=f"chemo_calibration_compare_CITOR.png")


# DL MODELS

for exp_name, model_name in zip(experiment_names, model_names):
    print("\nModel:", model_name)
    search_pattern = f"experiments/{exp_name}/**/{model_name}_ens_outputs.csv"
    #search_pattern = f"experiments/{exp_name}/**/{model_name}_outputs_external.csv"
    print(search_pattern)

    ensemble_predictions_dir = glob.glob(search_pattern, recursive=True)[0]

    ensemble_predictions = pd.read_csv(ensemble_predictions_dir, delimiter=";")
    print(ensemble_predictions_dir)
    ensemble_predictions["PatientID"] = [str(item).zfill(cfg.patient_id_length) for item in ensemble_predictions["PatientID"]]

    #ensemble_predictions = ensemble_predictions[["PatientID", "ensemble_prediction"]].copy()
    df_combined = pd.merge(ensemble_predictions, df_chemo_data, on="PatientID", how="left")

    chemo_ensemble_DL_predictions_list_dict = {}
    chemo_true_labels_list_dict = {}
    no_chemo_ensemble_DL_predictions_list_dict = {}
    no_chemo_true_labels_list_dict = {}

    
    
    for endpoint in endpoints:

        # drop all nones
        mask = df_combined[endpoint+"_true"].isin([0, 1])

        
        #mask2 = auc_no_chemo[endpoint+"_true"].isin([0, 1])

        df_combined_temp = df_combined[mask]

        total_auc = roc_auc_score(df_combined_temp[endpoint+"_true"], df_combined_temp[endpoint+"_pred"])
        print(f"Endpoint: {endpoint}, AUC total: {total_auc:.3f}")

        df_no_chemo_preds = df_combined_temp[df_combined["Chemotherapy"] == 0].copy()
        df_chemo_preds = df_combined_temp[df_combined["Chemotherapy"] == 1].copy()

        print(len(df_no_chemo_preds), len(df_chemo_preds))

        chemo_ensemble_DL_predictions_list_dict[endpoint] = torch.tensor(df_chemo_preds[endpoint+"_pred"].tolist())
        chemo_true_labels_list_dict[endpoint] = torch.tensor(df_chemo_preds[endpoint+"_true"].tolist())
        no_chemo_ensemble_DL_predictions_list_dict[endpoint] = torch.tensor(df_no_chemo_preds[endpoint+"_pred"].tolist())
        no_chemo_true_labels_list_dict[endpoint] = torch.tensor(df_no_chemo_preds[endpoint+"_true"].tolist())
        
        auc_no_chemo = roc_auc_score(df_no_chemo_preds[endpoint+"_true"], df_no_chemo_preds[endpoint+"_pred"])
        auc_chemo = roc_auc_score(df_chemo_preds[endpoint+"_true"], df_chemo_preds[endpoint+"_pred"])
        print(f"Endpoint: {endpoint}, AUC no chemotherapy: {auc_no_chemo:.3f}, AUC yes chemotherapy: {auc_chemo:.3f}")


    
    


    
    # make_calibration_plots(cfg, predictions_dict=no_chemo_ensemble_DL_predictions_list_dict, true_labels_dict=no_chemo_true_labels_list_dict, 
    #                                     lr_predictions_dict=chemo_ensemble_DL_predictions_list_dict, lr_true_labels_dict=chemo_true_labels_list_dict, mode='calibration', set_name="Ensemble (Test Set)", 
    #                                     #filename=os.path.join(cfg.exp_dir, 'calibration_plot_ensemble.png'))
    #                                     filename=f"with_chemo_calibration_compare_{model_name}.png")


print("\n\n\n\n")

    


    #ensemble_predictions = ensemble_predictions[ensemble_predictions["PatientID"].isin(chemo_patient_IDS)]
    #ensemble_predictions = ensemble_predictions[["PatientID", "ensemble_prediction"]].copy()
    #df_chemo_data = pd.merge(df_chemo_data, ensemble_predictions, on="PatientID", how="left")


CITOR MODEL
Endpoint: Aspiration_M06, AUC no chemotherapy: 0.830, AUC yes chemotherapy: 0.546
Endpoint: Dysphagia_M06, AUC no chemotherapy: 0.834, AUC yes chemotherapy: 0.760
Endpoint: Sticky_M06, AUC no chemotherapy: 0.763, AUC yes chemotherapy: 0.717
Endpoint: Taste_M06, AUC no chemotherapy: 0.727, AUC yes chemotherapy: 0.663
Endpoint: Xerostomia_M06, AUC no chemotherapy: 0.788, AUC yes chemotherapy: 0.730

Model: TransRP_ResNet18_m2
experiments/TransRP_ResNet18_m2 features/**/TransRP_ResNet18_m2_ens_outputs.csv
experiments/TransRP_ResNet18_m2 features\5_101_32_TransRP_ResNet18_m2_params_122913577_auc_tr_0.731_val_0.731_test_0.724_avg_tr_0.751_val_0.727_test_0.731_ens_0.738\TransRP_ResNet18_m2_ens_outputs.csv
Endpoint: Aspiration_M06, AUC total: 0.758
112 82
Endpoint: Aspiration_M06, AUC no chemotherapy: 0.782, AUC yes chemotherapy: 0.728
Endpoint: Dysphagia_M06, AUC total: 0.849
119 89
Endpoint: Dysphagia_M06, AUC no chemotherapy: 0.850, AUC yes chemotherapy: 0.846
Endpoint: Sticky_

C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_9956\903862219.py:46: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_no_chemo_preds = df_combined_temp[df_lr_combined["Chemotherapy"] == 0].copy()
C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_9956\903862219.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_chemo_preds = df_combined_temp[df_lr_combined["Chemotherapy"] == 1].copy()
C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_9956\903862219.py:46: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_no_chemo_preds = df_combined_temp[df_lr_combined["Chemotherapy"] == 0].copy()
C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_9956\903862219.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_chemo_preds = df_combined_temp[df_lr_combined["Chemotherapy"] == 1].copy()
C:\Users\d.c.macrae\AppData\Local\Temp\2\ipykernel_9956\903862219.py:46: UserW

In [15]:
a = torch.zeros(8,32,1,1,)
b = torch.ones(8,32,1,1,)

c = torch.concat([a,b], dim=1)
c.shape

torch.Size([8, 64, 1, 1])

In [1]:
# load ensemble predictions

experiment_names = ["Best_DCNN_ECE", "Best_ResNet_ECE", "Best_ViT", "Best ResNeXt"]
model_names = ["dcnn_pooling", "resnet_lrelu", "ViT", "resnext_lrelu"]

endpoints = ["Taste_M06"]
endpoints = config.endpoint_list


# LR MODEL
print("CITOR MODEL")
lr_ens_dir = glob.glob(f"final_models/{experiment_names[0]}/**/lr_ens_outputs.csv", recursive=True)[0]
df_lr_ens_preds = pd.read_csv(lr_ens_dir, delimiter=";")
df_lr_ens_preds["PatientID"] = [str(item).zfill(cfg.patient_id_length) for item in df_lr_ens_preds["PatientID"]]
df_lr_combined = pd.merge(df_lr_ens_preds, df_chemo_data, on="PatientID", how="left")


chemo_ensemble_DL_predictions_list_dict = {}
chemo_true_labels_list_dict = {}
no_chemo_ensemble_DL_predictions_list_dict = {}
no_chemo_true_labels_list_dict = {}

for endpoint in endpoints:
        # drop all nones
        mask = df_lr_combined[endpoint+"_true"].isin([0, 1])
        #mask2 = auc_no_chemo[endpoint+"_true"].isin([0, 1])

        df_combined_temp = df_lr_combined[mask]

        df_no_chemo_preds = df_combined_temp[df_combined["LEEFTIJD"] < 70].copy()
        df_chemo_preds = df_combined_temp[df_combined["LEEFTIJD"] >= 70 ].copy()

        chemo_ensemble_DL_predictions_list_dict[endpoint] = torch.tensor(df_chemo_preds[endpoint+"_pred"].tolist())
        chemo_true_labels_list_dict[endpoint] = torch.tensor(df_chemo_preds[endpoint+"_true"].tolist())
        no_chemo_ensemble_DL_predictions_list_dict[endpoint] = torch.tensor(df_no_chemo_preds[endpoint+"_pred"].tolist())
        no_chemo_true_labels_list_dict[endpoint] = torch.tensor(df_no_chemo_preds[endpoint+"_true"].tolist())
        
        auc_no_chemo = roc_auc_score(df_no_chemo_preds[endpoint+"_true"], df_no_chemo_preds[endpoint+"_pred"])
        auc_chemo = roc_auc_score(df_chemo_preds[endpoint+"_true"], df_chemo_preds[endpoint+"_pred"])
        print(f"Endpoint: {endpoint}, Under 70: {auc_no_chemo:.3f}, 70 and over: {auc_chemo:.3f}")

make_calibration_plots(cfg, predictions_dict=no_chemo_ensemble_DL_predictions_list_dict, true_labels_dict=no_chemo_true_labels_list_dict, 
                                        lr_predictions_dict=chemo_ensemble_DL_predictions_list_dict, lr_true_labels_dict=chemo_true_labels_list_dict, mode='calibration', set_name="Ensemble (Test Set)", 
                                        #filename=os.path.join(cfg.exp_dir, 'calibration_plot_ensemble.png'))
                                        filename=f"age_calibration_compare_CITOR.png")


# DL MODELS

for exp_name, model_name in zip(experiment_names, model_names):
    print("\nModel:", model_name)
    search_pattern = f"final_models/{exp_name}/**/{model_name}_ens_outputs.csv"
    #print(search_pattern)
    ensemble_predictions_dir = glob.glob(search_pattern, recursive=True)[0]

    ensemble_predictions = pd.read_csv(ensemble_predictions_dir, delimiter=";")
    print(ensemble_predictions_dir)
    ensemble_predictions["PatientID"] = [str(item).zfill(cfg.patient_id_length) for item in ensemble_predictions["PatientID"]]

    #ensemble_predictions = ensemble_predictions[["PatientID", "ensemble_prediction"]].copy()
    df_combined = pd.merge(ensemble_predictions, df_chemo_data, on="PatientID", how="left")

    chemo_ensemble_DL_predictions_list_dict = {}
    chemo_true_labels_list_dict = {}
    no_chemo_ensemble_DL_predictions_list_dict = {}
    no_chemo_true_labels_list_dict = {}

    
    
    for endpoint in endpoints:
        # drop all nones
        mask = df_combined[endpoint+"_true"].isin([0, 1])
        #mask2 = auc_no_chemo[endpoint+"_true"].isin([0, 1])

        df_combined_temp = df_combined[mask]

        df_no_chemo_preds = df_combined_temp[df_combined["LEEFTIJD"] < 70]
        df_chemo_preds = df_combined_temp[df_combined["LEEFTIJD"] >= 70]

        #print(len(df_chemo_preds))

        #print(df_no_chemo_preds.head())

        chemo_ensemble_DL_predictions_list_dict[endpoint] = torch.tensor(df_chemo_preds[endpoint+"_pred"].tolist())
        chemo_true_labels_list_dict[endpoint] = torch.tensor(df_chemo_preds[endpoint+"_true"].tolist())
        no_chemo_ensemble_DL_predictions_list_dict[endpoint] = torch.tensor(df_no_chemo_preds[endpoint+"_pred"].tolist())
        no_chemo_true_labels_list_dict[endpoint] = torch.tensor(df_no_chemo_preds[endpoint+"_true"].tolist())
        
        auc_no_chemo = roc_auc_score(df_no_chemo_preds[endpoint+"_true"], df_no_chemo_preds[endpoint+"_pred"])
        auc_chemo = roc_auc_score(df_chemo_preds[endpoint+"_true"], df_chemo_preds[endpoint+"_pred"])
        print(f"Endpoint: {endpoint}, Under 70: {auc_no_chemo:.3f}, 70 and over: {auc_chemo:.3f}")


    
    


    
    make_calibration_plots(cfg, predictions_dict=no_chemo_ensemble_DL_predictions_list_dict, true_labels_dict=no_chemo_true_labels_list_dict, 
                                        lr_predictions_dict=chemo_ensemble_DL_predictions_list_dict, lr_true_labels_dict=chemo_true_labels_list_dict, mode='calibration', set_name="Ensemble (Test Set)", 
                                        #filename=os.path.join(cfg.exp_dir, 'calibration_plot_ensemble.png'))
                                        filename=f"age_calibration_compare_{model_name}.png")


print("\n\n\n\n")

    


    #ensemble_predictions = ensemble_predictions[ensemble_predictions["PatientID"].isin(chemo_patient_IDS)]
    #ensemble_predictions = ensemble_predictions[["PatientID", "ensemble_prediction"]].copy()
    #df_chemo_data = pd.merge(df_chemo_data, ensemble_predictions, on="PatientID", how="left")


NameError: name 'config' is not defined

In [62]:
df_chemo_preds

,PatientID,Mode,Aspiration_M06_pred,Aspiration_M06_true,Dysphagia_M06_pred,Dysphagia_M06_true,Sticky_M06_pred,Sticky_M06_true,Taste_M06_pred,Taste_M06_true,Xerostomia_M06_pred,Xerostomia_M06_true,Modality_adjusted,Chemotherapy,Biotherapy
0,0052277,test,0.160359,0,0.419460,-1,0.379351,0,0.346174,1,0.448362,0,Chemoradiation,1,0
1,0066593,test,0.230783,0,0.369565,0,0.395063,0,0.361420,0,0.443817,0,Chemoradiation,1,0
2,0092560,test,0.180864,0,0.336398,0,0.368207,0,0.350371,0,0.415892,0,Chemoradiation,1,0
5,0260377,test,0.202054,0,0.418944,1,0.430300,1,0.405325,1,0.498054,1,Chemoradiation,1,0
6,0263641,test,0.160172,0,0.326240,0,0.360062,0,0.337092,1,0.419418,0,Chemoradiation,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,9300246,test,0.210483,0,0.406093,0,0.422674,0,0.400788,0,0.456072,1,Chemoradiation,1,0
211,9586209,test,0.170499,0,0.447674,1,0.392804,0,0.365513,0,0.450190,1,Chemoradiation,1,0
212,9597467,test,0.166129,0,0.363087,1,0.372850,0,0.354025,1,0.438901,1,Chemoradiation,1,0
214,9715913,test,0.199920,1,0.399997,1,0.427369,1,0.401445,1,0.493291,1,Chemoradiation,1,0


In [9]:
import os
import glob

exp_name = "Best_DCNN_ECE"
model_name = "dcnn_pooling"

search_pattern = f"final_models/{exp_name}/**/{model_name}_ens_outputs.csv"
ens_preds_dir = glob.glob(search_pattern, recursive=True)[0]

In [46]:
Chemo_merged_df = df.merge(df_chemo_data, on='PatientID', how='left')

Chemo_merged_df["Chemotherapy"] = Chemo_merged_df["Modality_adjusted"].str.contains("chemoradiation", case=False).astype(int)
Chemo_merged_df["Biotherapy"] = Chemo_merged_df["Modality_adjusted"].str.contains("bioradiation", case=False).astype(int)


train_val_df = Chemo_merged_df[Chemo_merged_df['Split']!="test"].copy()
test_df = Chemo_merged_df[Chemo_merged_df['Split']=="test"].copy()

In [47]:
print(train_val_df['Chemotherapy'].value_counts(normalize=True))

print(test_df['Chemotherapy'].value_counts(normalize=True))

Chemotherapy
0    0.654817
1    0.345183
Name: proportion, dtype: float64
Chemotherapy
0    0.582569
1    0.417431
Name: proportion, dtype: float64


In [75]:
# taste endpoints only
for endpoint in cfg.endpoint_list:
    print(endpoint)
    mask = train_val_df[endpoint].isin([0, 1])
    mask2 = test_df[endpoint].isin([0, 1])

    # Use.loc[] to keep only the rows where the mask is True
    train_val_df_taste = train_val_df.loc[mask]
    test_df_taste = test_df.loc[mask2]


    # train_val_df_taste = train_val_df.dropna(subset=[endpoint], inplace=False)
    # test_df_taste = test_df.dropna(subset=[endpoint], inplace=False)
    # test_df_taste = test_df.drop(~test_df[endpoint].isin([0, 1]), inplace=False)

    # test_df_taste = test_df.applymap(lambda x: np.nan if x not in [0, 1] else x)



    print(train_val_df_taste['Chemotherapy'].value_counts(normalize=True))

    print(test_df_taste['Chemotherapy'].value_counts(normalize=True))

    print(train_val_df_taste['Biotherapy'].value_counts(normalize=True))

    print(test_df_taste['Biotherapy'].value_counts(normalize=True))

    print(train_val_df_taste['OralCavity_Ext_meandose'].mean())

    print(test_df_taste['OralCavity_Ext_meandose'].mean())

    print(train_val_df_taste['Parotid_meandose'].mean())

    print(test_df_taste['Parotid_meandose'].mean())
    print()

Aspiration_M06
Chemotherapy
0    0.658974
1    0.341026
Name: proportion, dtype: float64
Chemotherapy
0    0.57732
1    0.42268
Name: proportion, dtype: float64
Biotherapy
0    0.958974
1    0.041026
Name: proportion, dtype: float64
Biotherapy
0    0.953608
1    0.046392
Name: proportion, dtype: float64
31.6567877264641
32.239356456458765
21.662044256657694
22.134523292814436

Dysphagia_M06
Chemotherapy
0    0.651659
1    0.348341
Name: proportion, dtype: float64
Chemotherapy
0    0.572115
1    0.427885
Name: proportion, dtype: float64
Biotherapy
0    0.956161
1    0.043839
Name: proportion, dtype: float64
Biotherapy
0    0.956731
1    0.043269
Name: proportion, dtype: float64
31.46995284605806
31.794657695860572
21.529524503700237
21.843195109697113

Sticky_M06
Chemotherapy
0    0.659004
1    0.340996
Name: proportion, dtype: float64
Chemotherapy
0    0.57868
1    0.42132
Name: proportion, dtype: float64
Biotherapy
0    0.957854
1    0.042146
Name: proportion, dtype: float64
Biotherap

In [74]:
# taste endpoints only

for chemo in ["Chemotherapy"]:
    print(endpoint)
    mask = train_val_df[chemo].isin([1])
    mask2 = test_df[chemo].isin([0])

    # Use.loc[] to keep only the rows where the mask is True
    train_val_df_taste = train_val_df.loc[mask]
    test_df_taste = test_df.loc[mask2]


    # train_val_df_taste = train_val_df.dropna(subset=[endpoint], inplace=False)
    # test_df_taste = test_df.dropna(subset=[endpoint], inplace=False)
    # test_df_taste = test_df.drop(~test_df[endpoint].isin([0, 1]), inplace=False)

    # test_df_taste = test_df.applymap(lambda x: np.nan if x not in [0, 1] else x)



    print(train_val_df_taste['Taste_M06'].value_counts(normalize=True))

    print(test_df_taste['Taste_M06'].value_counts(normalize=True))

    # print(train_val_df_taste['Biotherapy'].value_counts(normalize=True))

    # print(test_df_taste['Biotherapy'].value_counts(normalize=True))

    # print(train_val_df_taste['OralCavity_Ext_meandose'].mean())

    # print(test_df_taste['OralCavity_Ext_meandose'].mean())

    # print(train_val_df_taste['Parotid_meandose'].mean())

    # print(test_df_taste['Parotid_meandose'].mean())
    print()

Chemotherapy
Taste_M06
 0.0    0.641196
 1.0    0.242525
-1.0    0.116279
Name: proportion, dtype: float64
Taste_M06
 0.0    0.677165
 1.0    0.204724
-1.0    0.118110
Name: proportion, dtype: float64

